# CSCI 4380 - Data Mining (Final Project)
- Aiden Sagneri
- Anisha _____
- Carla _____
- Emily _____
- Joshua Mark
## Project Description
This project attempts to create a data mining model that can accept a dataset, that, by the use of smarthome sensors, record a resident's house-activity, and be able to reliably predict the resident's activity given up to 60 seconds of prior sensor data. To help generalize the model, this project uses two pre-existing datasets to help build the model instead of one: [Orange4Home](https://hal.science/hal-01639673) and [CASAS ____ ](https://pmc.ncbi.nlm.nih.gov/articles/PMC3886862/). The main purpose of this project is to first create this model, and then analyze how well it performs with the two pre-existind datasets, as well as note any interesting findings that were discovered in the process.

---
## Data Preprocessing
You should be able to just run these cells and then collapse the entire section.

In [32]:
# ================================================================================
# import packages
# ================================================================================
import os, math, time, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [33]:
# ================================================================================
# Import datasets
# ================================================================================
# Download the dataset from the web if it doesn't already exist in the current working directory
#
# Dataset (as DataFrame objects) are called:
# - o4h (orange4home dataset)
# - casas (CASAS dataset) 

# --- Orange4Home ---
O4H_DATA_URL = "https://github.com/JCumin/jcumin.github.io/raw/refs/heads/main/orange4home.zip" # Web location (downloads a .zip file)
O4H_ZIP_PATH = "orange4home.zip" # Import (.zip) of the web location 
O4H_CSV_PATH = os.path.join("orange4home", "o4h_all_events.csv") # Location in the .zip of the dataset (.csv file)

if not os.path.exists(O4H_CSV_PATH): # If .csv file not found, get it from the .zip file
    if not os.path.exists(O4H_ZIP_PATH): # If the .zip file does not exist, download from the web
        print("Downloading Orange4Home dataset (about 8 MB)...")
        urllib.request.urlretrieve(O4H_DATA_URL, O4H_ZIP_PATH)
    with zipfile.ZipFile(O4H_ZIP_PATH) as z: # "...get it from the .zip file"
        z.extract("orange4home/o4h_all_events.csv")
    os.remove(O4H_ZIP_PATH); # delete .zip after extracting dataset from it

# Read the dataset (.csv) file
o4h = pd.read_csv(O4H_CSV_PATH)
o4h['Time'] = pd.to_datetime(o4h['Time']) # convert time column to datetime objects
print(f"O4H: Number of events (rows): {o4h.shape[0]:,}")

del O4H_DATA_URL, O4H_ZIP_PATH, O4H_CSV_PATH

# --- CASAS ---
# TODO: Copy the above for CASAS
CASAS_DATA_URL = "" # TODO
CASAS_ZIP_PATH = "" # TODO (depends on how the data is stored on the web)
CASAS_CSV_PATH = "hh101.csv" # TODO (temp location)

# TODO
# if not.os.path.exists(CASAS_CSV_PATH)
# ...

# Read the dataset (.csv) file
casas = pd.read_csv(CASAS_CSV_PATH);
print(f"CASAS: Number of events (rows): {casas.shape[0]:,}")

del CASAS_DATA_URL, CASAS_ZIP_PATH, CASAS_CSV_PATH

O4H: Number of events (rows): 746,767
CASAS: Number of events (rows): 222,857


### Dataset Preprocessing
Format, parse, and/or manipulate the original datasets to get formal target variables as well as normalized input variables that the model can eventually use to be trained/tested on 

In [34]:
# ================================================================================
# Data Preprocessing
# ================================================================================

# --- Orange4Home ---
# Variables Saved:
# o4h_windows - a timeline (segmented into 60s windows) with the associated activity 
#   (label) for each window
# X_sensor - a timeline (segmented into 60s windows, each row is one window) that 
#   records the number of changes for each sensor
# o4h_y - the timeline from o4h_windows with only the labels (no start or end time columns)
# o4h_train - the first two weeks of o4h_windows
# o4h_test - the last two weeks of o4h_windows
#
# Other saved variables (idk what to do with them)
# minute, X_clock, X_log, X_bin


# Pull out the label events only (e.g. START:Office|Computing), sorted by time
lab = o4h[o4h['ItemName'] == 'label'].sort_values('Time').reset_index(drop=True)

# Split each label value into two parts:
kind = lab['Value'].str.split(':', n=1).str[0] # the "START" or "STOP" value
name = lab['Value'].str.split(':', n=1).str[1] # the activity being started/stopped (e.g. "Office|Computing")

# Separate labels start and stop tables (maintain sorted-chronology)
starts, stops = lab[kind == 'START'].reset_index(drop=True), lab[kind == 'STOP'].reset_index(drop=True)
assert (name[kind == 'START'].values == name[kind == 'STOP'].values).all() # double-check seperation is done correctly

# Recombine into list of labels with start and stop times (sorted by start-column)
intervals = pd.DataFrame({'Label': name[kind == 'START'].values, 'Start': starts['Time'].values, 'End': stops['Time'].values})
day_order = sorted(intervals['Start'].dt.date.unique())
intervals['DayIndex'] = intervals['Start'].dt.date.map({d: i for i, d in enumerate(day_order)})
intervals['Week'] = intervals['DayIndex'] // 5 + 1
# print(f"{len(o4h_intervals)} activity instances, {o4h_intervals['Label'].nunique()} place|activity labels, {len(day_order)} days")
# print(f"{o4h_intervals}")

WINDOW = '60s'

# Cut/expand every labeled interval into consecutive 60-second windows; a window 
# gets the label it lies in.
win_rows = []
for _, r in intervals.iterrows():
    for t in pd.date_range(r['Start'].floor(WINDOW), r['End'], freq=WINDOW):
        mid = t + pd.Timedelta(WINDOW) / 2
        if r['Start'] <= mid < r['End']:              # the window's midpoint is inside the activity
            win_rows.append((t, r['Label'], r['Week'], r['DayIndex']))
o4h_windows = pd.DataFrame(win_rows, columns=['WindowStart', 'Label', 'Week', 'DayIndex']).drop_duplicates('WindowStart')

# Create a new df where each cell counts the number of updates to a particular 
# sensor (column) is received during that particular window (row)
sensors = o4h[o4h['ItemName'] != 'label'].sort_values(['ItemName', 'Time'])
changes = sensors[sensors['Value'] != sensors.groupby('ItemName')['Value'].shift()].copy()
changes['WindowStart'] = changes['Time'].dt.floor(WINDOW)
counts = changes.groupby(['WindowStart', 'ItemName']).size().unstack(fill_value=0)
constant = sensors.groupby('ItemName')['Value'].nunique() == 1   # 28 sensors never change: no information
sensor_cols = [c for c in counts.columns if not constant[c]]
X_sensor = counts.reindex(o4h_windows['WindowStart'], fill_value=0)[sensor_cols].reset_index(drop=True)

# I'm not sure what this is used for (Copied from hw 3).
# "3) Clock features (kept separate, to measure how much the schedule alone gives away)"
minute = o4h_windows['WindowStart'].dt.hour * 60 + o4h_windows['WindowStart'].dt.minute
X_clock = pd.DataFrame({'clock_sin': np.sin(2 * np.pi * minute / 1440),
                        'clock_cos': np.cos(2 * np.pi * minute / 1440)})

# Finally, create normalized datasets
o4h_y = o4h_windows['Label'].reset_index(drop=True)
o4h_train = (o4h_windows['Week'] <= 2).values   # weeks 1-2 for training
o4h_test = ~o4h_train                       # weeks 3-4 for testing

# Once again, i'm not sure how this is used (copied from hw 3).
X_log = np.log1p(X_sensor)          # change counts are very skewed; log keeps big counts from dominating
X_bin = (X_sensor > 0).astype(int)  # "did this sensor change in the window?" for ID3 and Naive Bayes

print(f"Windows: {len(o4h_windows):,} (train {o4h_train.sum():,}, test {o4h_test.sum():,}), sensor features: {len(sensor_cols)}")
print('\nWindows per label (train / test):')
print(pd.crosstab(o4h_y, np.where(o4h_train, 'train', 'test'))[['train', 'test']].sort_values('train').to_string())

del lab, kind, name, starts, stops, intervals, day_order, win_rows, sensors, changes, counts, constant, sensor_cols

Windows: 10,311 (train 5,282, test 5,029), sensor features: 208

Windows per label (train / test):
col_0                       train  test
Label                                  
Kitchen|Cleaning                3     5
Bathroom|Cleaning               4     5
Bedroom|Cleaning                4     2
Office|Cleaning                 5     5
Entrance|Leaving                7    10
Toilet|Using_the_toilet         8     8
Living_room|Cleaning            9    10
Staircase|Going_down           10     5
Staircase|Going_up             10     8
Bedroom|Dressing               14    16
Kitchen|Preparing              17    13
Entrance|Entering              19    14
Bathroom|Using_the_toilet      23    12
Kitchen|Washing_the_dishes     58    48
Bathroom|Using_the_sink        60    58
Kitchen|Cooking               115    94
Living_room|Eating            126    93
Office|Watching_TV            144   116
Bathroom|Showering            174   155
Bedroom|Napping               197   203
Living_room|Computing

In [35]:
# --- CASAS ---
# TODO: copy the above to create a target dataframe like o4h_intervals

### Data Model
The classes and functions that represent the learning model and its behaviors.

#### Utility Functions

In [36]:
def count_items(items):
    counts = {}
    for x in items:
        counts[x] = counts.get(x, 0) + 1
    return counts

def class_probabilities(labels):
    """
       Takes a 1D list of event outcomes (y).

       Returns a 1D list where each cell represents the probability of each unique outcome listed in labels (that occured in y)
    """
    total = float(len(labels))
    counts = count_items(labels)
    return [counts[label] / total for label in counts]

def entropy(probabilities):
    """ 
        Returns the entropy of (the average number of bits it takes to represent an) event with the given probabilities.
    """
    return sum(-p * math.log2(p) for p in probabilities if p > 0)

def data_entropy(labels):
    """
        Returns the entropy of (the average number of bits it takes to represent) the given events
    """
    return entropy(class_probabilities(labels))

def gini_impurity(labels):
    """
    DONE: Calculate the Gini impurity of a list of labels.
    Formula: 1 - sum((p_i)^2)
    """
    p = class_probabilities(labels);
    p2 = [i ** 2 for i in p];
    s = sum(p2);
    return 1 - s;

def _same_category(left, right):
    # Treat a missing floating-point category as equal to another missing value.
    if left == right:
        return True;
    return left != left and right != right;

def weighted_gini(X, y, attribute, split_value):
    """
    DONE: Calculate the weighted Gini impurity of a binary split.
    The split is defined by: x[attribute] == split_value (True branch)
                             x[attribute] != split_value (False branch)
    """
    # A tuple allows the true branch to contain a subset of a categorical feature's values.
    split_values = split_value if isinstance(split_value, tuple) else (split_value,);
    t_branch = [
        label for row, label in zip(X, y)
        if any(_same_category(row[attribute], value) for value in split_values)
    ];
    f_branch = [
        label for row, label in zip(X, y)
        if not any(_same_category(row[attribute], value) for value in split_values)
    ];

    total = len(X);
    t_ratio = len(t_branch) / total;
    f_ratio = len(f_branch) / total;

    return (t_ratio * gini_impurity(t_branch)) + (f_ratio * gini_impurity(f_branch));

def bootstrap_sample(X, y):
    """
    Generate a bootstrap sample from a training dataset.

    A bootstrap sample is constructed by randomly selecting observations WITH replacement until the generated sample
    contains the same number of observations as the original dataset.

    Parameters
    ----------
    X : pandas.DataFrame
        Training features.

    y : pandas.Series
        Training labels.

    Returns
    -------
    X_boot : pandas.DataFrame
        Bootstrap-sampled feature matrix.

    y_boot : pandas.Series
        Bootstrap-sampled labels.

    indices : list
        Zero-based row positions selected from the input, used for OOB evaluation.

    Notes
    -----
    - Duplicate observations are expected.
    - Some observations may not appear in the sample.
    - The returned indices will later be used for Out-of-Bag (OOB) evaluation.
    """

    if len(X) != len(y):
        raise ValueError("X and y must contain the same number of observations.")
    if len(X) == 0:
        raise ValueError("Cannot bootstrap an empty dataset.")

    indices = np.random.choice(len(X), size=len(X), replace=True).tolist()
    X_boot = X.iloc[indices].reset_index(drop=True)
    y_boot = y.iloc[indices].reset_index(drop=True)

    return X_boot, y_boot, indices

#### Model Classes
The main class is CARTTree, which is a decision tree that splits each node according to Gini Impurity, with a specific control for decision tree depth. RandomForestTree extends this CARTTree with changes to randomly select the attribute (from a set of candidates) to split the node on; and RandomForest implements multiple instances of RandomForestTree to aggregate multiple decision trees and combine the results/final behaviors. 

In [37]:
class CARTTree:
    class _Leaf:
        def __init__(self, value):
            self.value = value

        def predict(self, x):
            return self.value

    class _BinarySplit:
        def __init__(self, attribute, split_value, true_branch, false_branch, default_value):
            self.attribute = attribute
            self.split_value = split_value
            self.true_branch = true_branch
            self.false_branch = false_branch
            self.default_value = default_value

        def predict(self, x):
            # DONE: Implement the prediction traversal for a binary split.
            # If x[self.attribute] == self.split_value, go to true_branch.
            # Otherwise, go to false_branch. Handle missing attributes safely.
            # Missing attributes fall back to the majority class stored at this node.
            if self.attribute not in x: return self.default_value;
            
            split_values = self.split_value if isinstance(self.split_value, tuple) else (self.split_value,);
            
            if any(_same_category(x[self.attribute], value) for value in split_values):
                return self.true_branch.predict(x);
            return self.false_branch.predict(x);

    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1, min_impurity_decrease=0.0):
        self.root = None
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_impurity_decrease = min_impurity_decrease

    def fit(self, X, y):
        attributes = list(X[0].keys())
        self.root = self._build_tree(X, y, attributes, depth=0)

    def predict(self, x):
        if self.root is None:
            raise ValueError("Tree not trained")
        return self.root.predict(x)

    def predict_all(self, X):
        # Apply the single-example prediction method to every feature row.
        return [self.predict(x) for x in X];

    def accuracy(self, X, y):
        correct = sum(1 for x_i, y_i in zip(X, y) if self.predict(x_i) == y_i)
        return correct / len(y)

    def _partition_by(self, X, y, attribute, split_value):
        """
        DONE: Partition the dataset into two subsets (True and False) 
        based on whether x[attribute] == split_value.
        Returns: (X_true, y_true), (X_false, y_false)
        """
        # Preserve feature-row and target-label alignment in both branches.
        split_values = split_value if isinstance(split_value, tuple) else (split_value,);
        X_true = [];
        y_true = [];
        X_false = [];
        y_false = [];
        for row, label in zip(X, y):
            is_true = any(_same_category(row[attribute], value) for value in split_values);
            if is_true:
                X_true.append(row);
                y_true.append(label);
            else:
                X_false.append(row);
                y_false.append(label);
        return (X_true, y_true), (X_false, y_false);

    def _select_candidate_attributes(self, attributes):
        return attributes

    def _build_tree(self, X, y, attributes, depth):
        """
        DONE: Implement the recursive tree-building logic with stopping criteria.
        
        Stopping Criteria Checklist:
        1. Are all labels identical? (Pure node)
        2. Is depth >= self.max_depth?
        3. Is len(y) < self.min_samples_split?
        If any of the above are true, return a _Leaf.

        Split Logic:
        1. Calculate current node's Gini impurity.
        2. Find best attribute and split_value that minimizes weighted Gini.
        3. Calculate impurity decrease: (current_gini - weighted_gini).
        
        Post-Split Checks:
        1. Is impurity decrease < self.min_impurity_decrease?
        2. Does either the true or false branch have fewer than self.min_samples_leaf rows?
        If either is true, cancel the split and return a _Leaf.
        
        Otherwise, recursively build true and false branches and return a _BinarySplit.
        """
        # --- Stopping Criterea ---
        # Use the node's majority label for stopping leaves and unseen-value fallback.
        label_counts = count_items(y);
        majority_label = max(label_counts, key=label_counts.get);

        # Stop on pure labels, depth limit, or insufficient samples.
        if (len(label_counts) == 1) or (self.max_depth is not None and depth >= self.max_depth) or (len(y) < self.min_samples_split):
            return self._Leaf(majority_label);

        # --- Split Logic ---
        # Find every non-duplicate binary grouping of the observed categories.
        best_attr, best_split_val, best_w_gini = None, None, float("inf");
        for attribute in self._select_candidate_attributes(attributes):
            categories = [];
            for row in X:
                value = row[attribute];
                if not any(_same_category(value, seen) for seen in categories):
                    categories.append(value);
            if len(categories) < 2: continue;

            # Fix the first value on the true side so complements are scored only once.
            for subset_size in range(len(categories) - 1):
                for other_values in itertools.combinations(categories[1:], subset_size):
                    split_values = (categories[0],) + other_values;
                    split_gini = weighted_gini(X, y, attribute, split_values);
                    if split_gini < best_w_gini:
                        best_attr = attribute;
                        best_split_val = split_values;
                        best_w_gini = split_gini;

        # Stop if all remaining attributes are constant at this node.
        if best_attr is None:
            return self._Leaf(majority_label);

        current_gini = gini_impurity(y);
        impurity_decrease = current_gini - best_w_gini;
        (X_true, y_true), (X_false, y_false) = self._partition_by(
            X, y, best_attr, best_split_val
        );

        # --- Post-Split Checks ---
        # Reject splits that fail the configured impurity or child-size thresholds.
        if impurity_decrease < self.min_impurity_decrease or len(y_true) < self.min_samples_leaf or len(y_false) < self.min_samples_leaf:
            return self._Leaf(majority_label);

        # Reuse available attributes in both recursive branches, as CART permits.
        true_branch = self._build_tree(X_true, y_true, attributes, depth + 1);
        false_branch = self._build_tree(X_false, y_false, attributes, depth + 1);
        return self._BinarySplit(
            best_attr, best_split_val, true_branch, false_branch, majority_label
        );

In [38]:
class RandomForestTree(CARTTree):
    """
    Extension of the CARTTree implementation that supports random feature selection during split evaluation.

    Expected Behavior
    -----------------
    When searching for the best split at a node, evaluate only a randomly selected subset of candidate attributes rather than all available attributes.
    """

    def __init__(self, max_features="sqrt", **kwargs):
        """
        Parameters
        ----------
        max_features : int, str, or None
            Controls how many attributes are considered when searching for the best split.

        **kwargs
            Additional parameters passed to CARTTree.
        """
        self.max_features = max_features
        super().__init__(**kwargs)

    def _select_candidate_attributes(self, attributes):
        """
        Select the attributes that will be evaluated for splitting at the current node.

        Parameters
        ----------
        attributes : list
            Available attributes at the current node.

        Returns
        -------
        list
            Randomly selected candidate attributes.
        """
        return select_random_features(attributes, self.max_features)

In [39]:
class RandomForest:
    """
    Random Forest classifier built using your CART implementation.

    Each tree must be trained using:

    1. A bootstrap sample of the training data.
    2. Random feature selection during split evaluation.

    Final predictions should be generated using majority voting across all trees in the ensemble.
    """

    def __init__(
        self,
        n_trees=10,
        max_depth=None,
        max_features="sqrt"
    ):
        """
        Parameters
        ----------
        n_trees : int
            Number of trees in the forest.

        max_depth : int or None
            Maximum depth of each tree.

        max_features : int, str, or None
            Feature-selection strategy passed to each RandomForestTree.
        """
        if not isinstance(n_trees, int) or isinstance(n_trees, bool) or n_trees < 1:
            raise ValueError("n_trees must be a positive integer.")
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.max_features = max_features
        self.trees = []
        self.bootstrap_indices = []

    def fit(self, X, y):
        """
        Train the Random Forest.

        Requirements
        ------------
        For each tree:

        1. Create a bootstrap sample.
        2. Train a RandomForestTree on that sample.
        3. Store the tree for future predictions.

        Consider storing bootstrap indices to support Out-of-Bag (OOB) evaluation.
        """
        if len(X) != len(y):
            raise ValueError("X and y must contain the same number of observations.")
        if len(X) == 0:
            raise ValueError("Cannot fit a random forest on an empty dataset.")

        self.trees = []
        self.bootstrap_indices = []
        for _ in range(self.n_trees):
            X_boot, y_boot, indices = bootstrap_sample(X, y)
            tree = RandomForestTree(max_features=self.max_features, max_depth=self.max_depth)
            tree.fit(X_boot.to_dict(orient="records"), y_boot.tolist())
            self.trees.append(tree)
            self.bootstrap_indices.append(indices)
        return self

    def predict(self, X):
        """
        Predict class labels for a dataset.

        Requirements
        ------------
        1. Obtain predictions from every tree.
        2. Use majority voting to determine the final prediction for each observation.

        Returns
        -------
        list
            Predicted labels.
        """
        if not self.trees:
            raise ValueError("The random forest has not been fitted yet.")
        records = X.to_dict(orient="records") if isinstance(X, pd.DataFrame) else list(X)
        if not records:
            return []
        tree_predictions = [tree.predict_all(records) for tree in self.trees]
        return [
            self.majority_vote([predictions[i] for predictions in tree_predictions])
            for i in range(len(records))
        ]

    def majority_vote(self, predictions):
        """
        Determine the most frequently predicted class.

        Parameters
        ----------
        predictions : list
            Predictions collected from all trees.

        Returns
        -------
        object
            Majority class label.
        """
        if not predictions:
            raise ValueError("Cannot vote on an empty set of predictions.")
        return Counter(predictions).most_common(1)[0][0]

---
## Data Mining
Creating an instance of the data model and training it on the preprocessed data (Orange4Home and CASAS____)

In [40]:
# ================================================================================
# Data Preprocessing
# ================================================================================

# --- Orange4Home ---
o4h_part_i = int(X_sensor.shape[0]*0.8) # the index to partition o4h into training and testing sets

# training set should be 80% of total data
o4h_train_X = X_sensor.iloc[:o4h_part_i] # pull input from X_sensor
o4h_train_y = o4h_y.iloc[:o4h_part_i] # pull output from o4h_y

# testing set should be the remaining 20% of total data
o4h_test_X = X_sensor.iloc[o4h_part_i:]
o4h_test_y = o4h_y.iloc[o4h_part_i:]

# build, train, and test model
model = RandomForest() # initializing
model.fit(o4h_train_X, o4h_train_y) # training
# model.predict(o4h_test_y)

NameError: name 'select_random_features' is not defined

---
## Analysis and Conclusions